# Federated Learning Client — Glaucoma Research Centre

This notebook implements the **Data Scientist (client) side** of a PySyft federated learning workflow.

The pipeline:
1. Connect to the remote data site and retrieve the dataset asset
2. Define a remote function that preprocesses data and trains a CNN
3. Submit a code request for Data Owner approval
4. Execute the approved function and download results
5. Save / reload model weights for subsequent training rounds
6. Shut down the local orchestration server

## 1. Setup — Connect to the Data Site

In [1]:
import syft as sy
print(sy.__version__)

0.9.5


In [2]:
# Launch a local orchestration server and connect to the remote data site
data_site = sy.orchestra.launch(name="Glaucoma-research-centre")

CRITICAL:syft.server.server:Hash of the signing key '5e298...'


Using SQLiteDBConfig and sqlite:///C:\Users\ruben\AppData\Local\Temp\syft\c0ac1c82615848859d8d7ae2e5ee1548\db\c0ac1c82615848859d8d7ae2e5ee1548_json.db


SyftInfo: You have launched a development server at http://0.0.0.0:None. It is intended only for local use.

In [3]:
# Authenticate as the data scientist
client = data_site.login(email="rachel@datascience.inst", password="syftrocks")

Logged into <Glaucoma-research-centre: High side Datasite> as <rachel@datascience.inst>


## 2. Retrieve the Dataset Asset

In [4]:
# Access the protected dataset and the specific image asset
dataset = client.datasets["Medical Images Dataset"]
asset = dataset.assets["classification_images"]

In [5]:
# Inspect the mock pointer — shows the data structure without
# accessing real patient data on the server
data_pointer = asset.pointer
print("Mock data pointer structure:", data_pointer)

Mock data pointer structure: {'images': array([[[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        ...,

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]],


       [[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0

## 3. Create a Project and Define the Remote Function

The function below runs **entirely on the server** — it preprocesses the images and trains a `SimpleCNN`. Only the aggregated results (loss history + model weights) are returned to the client.

In [6]:
# Create a PySyft Project to group code requests
my_project = sy.Project(
    name="My TFG Project",
    description="Training a classification model using remote data preprocessing.",
    members=[client]
)

In [ ]:
from syft.service.policy.policy import MixedInputPolicy

# MixedInputPolicy allows combining a protected asset (raw_data_dict)
# with a plain Python type (previous_weights) in a single remote call.
@sy.syft_function(
    input_policy=MixedInputPolicy(
        client=client,
        raw_data_dict=asset,
    )
)
def remote_preprocessing_and_training(raw_data_dict):

    try:


        # --- Return results ---
        return {
            "status": "Success",

            "error": None
        }

    except Exception:
        return {"status": "ERROR", "error": "error"}

SyftSuccess: Syft function 'remote_preprocessing_and_training' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

## 4. Submit the Code Request

This sends the function to the Data Owner for review. Execution will not happen until it is approved.

In [8]:
my_project.create_code_request(obj=remote_preprocessing_and_training, client=client)
print("Project created and code execution requested. Waiting for Data Owner approval...")

Project created and code execution requested. Waiting for Data Owner approval...


## 5. Execute the Approved Function

Once the Data Owner approves the request, re-authenticate and run the function.

In [11]:
# Re-authenticate to pick up the approved code request
client = data_site.login(email="rachel@datascience.inst", password="syftrocks")

Logged into <Glaucoma-research-centre: High side Datasite> as <rachel@datascience.inst>


In [12]:
# Retrieve the approved remote function from the server
approved_function = client.code.remote_preprocessing_and_training

### Round 1 — Train from scratch

In [13]:
# Trigger execution on the remote server (no previous weights for round 1)
result_pointer = approved_function(raw_data_dict=asset)
print("Code is executing on the remote server...")

Error: 'previous_weights'
Traceback (most recent call last):
  File "c:\Users\ruben\anaconda3\envs\new_env\lib\site-packages\syft\service\policy\policy.py", line 543, in filter_kwargs
    passed_id = kwargs[kw]
KeyError: 'previous_weights'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\ruben\anaconda3\envs\new_env\lib\site-packages\syft\server\server.py", line 1186, in handle_api_call_with_unsigned_result
    result = method(context, *api_call.args, **api_call.kwargs)
  File "c:\Users\ruben\anaconda3\envs\new_env\lib\site-packages\syft\service\service.py", line 490, in _decorator
    result = func(self, *args, **kwargs)
  File "c:\Users\ruben\anaconda3\envs\new_env\lib\site-packages\syft\service\code\user_code_service.py", line 423, in call
    return self._call(context, uid, **kwargs).unwrap()
  File "c:\Users\ruben\anaconda3\envs\new_env\lib\site-packages\syft\types\result.py", line 90, in unwrap
    raise se

In [ ]:
# Download the results once execution is complete
final_result = result_pointer.get()
print("Execution finished! Results:")
print("Status       :", final_result["status"])
print("Total images :", final_result["total_images"])
print("Loss history :", final_result["loss_history"])

In [32]:
import torch

# Save round-1 weights to disk for use in the next training round
round1_weights = final_result["model_weights"]
torch.save(round1_weights, "weights_round_1.pth")
print("Round-1 weights saved to disk!")

### Round 2 — Continue training from Round 1 weights

In [14]:
import torch

# Load round-1 weights and convert tensors to plain Python lists.
# PySyft's MixedInputPolicy requires JSON-serialisable types for non-asset kwargs.
round1_weights = torch.load("weights_round_1.pth")
weights_for_syft = {k: v.tolist() for k, v in round1_weights.items()}

# Execute round 2 with the previous weights
result_pointer = approved_function(
    raw_data_dict=asset,
    previous_weights=weights_for_syft
)
print("Code is executing on the remote server...")

In [31]:
# Download round-2 results
final_result = result_pointer.get()
print("Execution finished! Results:")
print("Status       :", final_result["status"])
print("Total images :", final_result["total_images"])
print("Loss history :", final_result["loss_history"])

In [ ]:
# Save round-2 weights for any further training rounds
round2_weights = final_result["model_weights"]
torch.save(round2_weights, "weights_round_2.pth")
print("Round-2 weights saved to disk!")

## 6. Shut Down

In [36]:
# Stop the local PySyft orchestration server
data_site.land()